In [4]:
import pandas as pd


df_row = pd.read_csv("FoodFacts.products.170k_May26.csv")



FileNotFoundError: [Errno 2] No such file or directory: 'FoodFacts.products.170k_May26.csv'

In [ ]:
print("Shape:", df_row.shape)

print("\nColumns:")
print(df_row.columns.tolist())

print("\nTarget Distribution:")
print(df_row["ecoscore_grade"].value_counts(dropna=False))

In [ ]:
missing = (
    df_row.isnull().sum()
    .sort_values(ascending=False)
)

print(missing.head(30))

In [ ]:
print(df_row['categories'].head(10))

In [ ]:
print(df_row['categories'].sample(10, random_state=42))

In [ ]:
df_row['categories'].isnull().sum()

In [ ]:
ingredient_cols = [
    c for c in df_row.columns
    if 'ingredients_hierarchy[' in c
]

In [ ]:
category_cols = [
    c for c in df_row.columns
    if 'categories_hierarchy[' in c
]

In [ ]:
print(len(ingredient_cols))
print(len(category_cols))

In [ ]:
def merge_hierarchy(row):

    values = []

    for item in row:

        if pd.notnull(item):

            item = str(item)

            item = item.replace('en:', '')
            item = item.replace('-', ' ')
            item = item.strip()

            if item:
                values.append(item)

    values = list(dict.fromkeys(values))

    return ' '.join(values)

In [ ]:
df_row['ingredient_hierarchy_text'] = (
    df_row[ingredient_cols]
    .apply(merge_hierarchy, axis=1)
)
df_row['category_hierarchy_text'] = (
    df_row[category_cols]
    .apply(merge_hierarchy, axis=1)
)

In [ ]:
print(df_row['ingredient_hierarchy_text'].head())
print("*"*50)
print(df_row['category_hierarchy_text'].head())

In [ ]:
print(
    df_row[
        [
            'product_name_en',
            'ingredients_text_en',
            'categories',
            'ingredient_hierarchy_text',
            'category_hierarchy_text'
        ]
    ].isnull().sum()
)

In [ ]:
(
    df_row['ingredient_hierarchy_text']
    .str.strip()
    .eq('')
    .sum()
)

In [5]:
(
    df_row['category_hierarchy_text']
    .str.strip()
    .eq('')
    .sum()
)

NameError: name 'df_row' is not defined

In [ ]:
df_row["_id"].value_counts()

In [6]:
(
    (
        df_row['product_name_en'].fillna('').str.strip() == ''
    ) &
    (
        df_row['ingredients_text_en'].fillna('').str.strip() == ''
    ) &
    (
        df_row['categories'].fillna('').str.strip() == ''
    ) &
    (
        df_row['ingredient_hierarchy_text'].fillna('').str.strip() == ''
    ) &
    (
        df_row['category_hierarchy_text'].fillna('').str.strip() == ''
    )
).sum()

NameError: name 'df_row' is not defined

In [ ]:
df_row[['_id', 'product_name_en']].head(20)

In [ ]:
df_nlp = df_row[
    [
        'ecoscore_grade',
        'product_name_en',
        'ingredients_text_en',
        'categories',
        'ingredient_hierarchy_text',
        'category_hierarchy_text'
    ]
].copy()

In [ ]:
df_nlp[
    df_nlp['ingredient_hierarchy_text']
    .astype(str)
    .str.fullmatch(r'e\d+[a-z]*', na=False)
].shape

In [ ]:
df_row[
    df_row['ingredient_hierarchy_text']
    .astype(str)
    .str.fullmatch(r'[\d\.Ee\+\-]+', na=False)
][['_id', 'ingredient_hierarchy_text']].head(20)

In [7]:
(
    df_row["ingredient_hierarchy_text"]
    .fillna('')
    .astype(str)
    .str.fullmatch(r'[\d\.Ee\+\-]+')
    .sum()
)

NameError: name 'df_row' is not defined

In [ ]:
df_row = df_row[
    ~(
        (
            df_row['product_name_en'].fillna('').str.strip() == ''
        ) &
        (
            df_row['ingredients_text_en'].fillna('').str.strip() == ''
        ) &
        (
            df_row['categories'].fillna('').str.strip() == ''
        ) &
        (
            df_row['ingredient_hierarchy_text'].fillna('').str.strip() == ''
        ) &
        (
            df_row['category_hierarchy_text'].fillna('').str.strip() == ''
        )
    )
].copy()

In [ ]:
print(df_nlp.shape)

In [ ]:
print(df_nlp.sample(5, random_state=42))

In [ ]:
df_nlp

In [ ]:
print(df_nlp['product_name_en'].sample(10, random_state=1).tolist())

In [ ]:
print(df_nlp['ingredients_text_en'].sample(10, random_state=1).tolist())

In [ ]:
print(df_nlp['categories'].sample(10, random_state=1).tolist())

In [ ]:
print(df_nlp[['categories',
              'category_hierarchy_text']].sample(
                  10,
                  random_state=42
              ))

In [ ]:
df_nlp = df_row[
    [
        'ecoscore_grade',
        'product_name_en',
        'ingredients_text_en',
        'ingredient_hierarchy_text',
        'category_hierarchy_text'
    ]
].copy()

In [ ]:
print(df_nlp.shape)

print(df_nlp.isnull().sum())

In [8]:
(
    (
        df_nlp['product_name_en'].fillna('').str.strip() == ''
    ) &
    (
        df_nlp['ingredients_text_en'].fillna('').str.strip() == ''
    ) &
    (
        df_nlp['ingredient_hierarchy_text'].fillna('').str.strip() == ''
    ) &
    (
        df_nlp['category_hierarchy_text'].fillna('').str.strip() == ''
    )
).sum()

NameError: name 'df_nlp' is not defined

In [ ]:
import re
import unicodedata

def clean_text_lstm(text):

    if pd.isna(text):
        return ''

    text = str(text)

    # unicode normalization
    text = unicodedata.normalize('NFKC', text)

    # remove language tags
    text = re.sub(
        r'\b(?:en|fr|de|pt|it|ru|es|nl):',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # remove urls
    text = re.sub(
        r'http\S+|www\S+',
        ' ',
        text
    )

    # replace hyphens
    text = text.replace('-', ' ')

    # remove punctuation
    text = re.sub(
        r'[^\w\s]',
        ' ',
        text
    )

    # normalize spaces
    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    # lowercase
    text = text.lower()

    return text

In [ ]:
text_cols = [
    'product_name_en',
    'ingredients_text_en',
    'ingredient_hierarchy_text',
    'category_hierarchy_text'
]

for col in text_cols:
    df_nlp[col] = (
        df_nlp[col]
        .fillna('')
        .apply(clean_text_lstm)
    )

In [ ]:
print(df_nlp.sample(10, random_state=42))

In [ ]:
print(df_nlp.isnull().sum())

In [ ]:
for col in [
    'product_name_en',
    'ingredients_text_en',
    'ingredient_hierarchy_text',
    'category_hierarchy_text'
]:

    non_empty = (
        df_nlp[col]
        .str.strip()
        .ne('')
        .sum()
    )

    print(col)
    print(non_empty)
    print(
        round(
            non_empty / len(df_nlp) * 100,
            2
        ),
        '%'
    )
    print()

In [ ]:
df_nlp['deep_text'] = (
    df_nlp['product_name_en'] + ' ' +
    df_nlp['ingredients_text_en'] + ' ' +
    df_nlp['ingredient_hierarchy_text'] + ' ' +
    df_nlp['category_hierarchy_text']
)

In [ ]:
print(df_nlp['deep_text'].sample(10, random_state=42))

In [ ]:
df_nlp['deep_text'].str.split().apply(len).describe()

In [ ]:
lengths = (
    df_nlp['deep_text']
    .str.split()
    .apply(len)
)

print("90% :", lengths.quantile(0.90))
print("95% :", lengths.quantile(0.95))
print("99% :", lengths.quantile(0.99))

In [ ]:
df_nlp.loc[
    lengths > 500,
    'deep_text'
].head(5)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [ ]:
label_encoder = LabelEncoder()

df_nlp['label'] = label_encoder.fit_transform(
    df_nlp['ecoscore_grade']
)

print(dict(zip(label_encoder.classes_,label_encoder.transform(   label_encoder.classes_) )))

In [ ]:
X = df_nlp['deep_text']

y = df_nlp['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts())

### Text Vectorization: Bag-of-Words

We will use `CountVectorizer` to convert text data into numerical feature vectors using the Bag-of-Words (BoW) model. This approach counts the occurrences of words within a document, creating a vector where each dimension corresponds to a unique word in the vocabulary.

`max_features` controls the size of the vocabulary, `ngram_range` allows capturing combinations of words (bigrams in this case), and `min_df` filters out words that appear too infrequently.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Bag of Words (BoW) Vectorization
# max_features: Limits the vocabulary size to the most frequent words.
# ngram_range: Includes unigrams and bigrams for better context capture.
# min_df: Ignores terms that appear in less than 2 documents to reduce noise.
bow_vectorizer = CountVectorizer(
    max_features=100000,
    ngram_range=(1,2),
    min_df=2
)

# Fit the vectorizer on the training data and transform both training and test data
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


### Handling Imbalanced Data with SMOTE

Our target variable `ecoscore_grade` might have an imbalanced distribution, which can negatively impact model performance. To address this, we use **SMOTE (Synthetic Minority Over-sampling Technique)**.

SMOTE works by creating synthetic samples from the minority class, helping to balance the class distribution. This prevents models from being biased towards the majority class and improves their ability to classify minority classes correctly.

In [ ]:
from imblearn.over_sampling import SMOTE

# Initialize SMOTE with a fixed random state for reproducibility
smote = SMOTE(random_state=42)

# Apply SMOTE to the Bag-of-Words training data
# This will generate synthetic samples for minority classes to balance the dataset
X_train_bow_smote, y_train_bow_smote = smote.fit_resample(
    X_train_bow,
    y_train
)

print("Shape of X_train_bow after SMOTE:", X_train_bow_smote.shape)
print("Distribution of y_train_bow after SMOTE:")
print(pd.Series(y_train_bow_smote).value_counts())


### Text Classification Models

This section defines a class `TextClassificationModels` to train and evaluate various machine learning models. As requested, we will focus on a subset of models: **Naive Bayes**, **Random Forest**, and **XGBoost**. These models are well-suited for text classification tasks.

-   **Naive Bayes (MultinomialNB)**: A probabilistic classifier particularly effective for text data, assuming feature independence.
-   **Random Forest (RandomForestClassifier)**: An ensemble learning method that builds multiple decision trees and merges their predictions for improved accuracy and robustness.
-   **XGBoost (XGBClassifier)**: A powerful gradient boosting framework known for its speed and performance, especially on structured and tabular data, which can also be applied to text features.

Each model is trained on the preprocessed training data (`X_train_bow_smote`, `y_train_bow_smote`) and evaluated on the test set (`X_test_bow`, `y_test`) using accuracy and a detailed classification report.

In [9]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import classification_report, accuracy_score


class TextClassificationModels:

    def __init__(self):
        self.models = {
            "Naive Bayes": MultinomialNB(),
            "Random Forest": RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1 # Use all available cores for faster training
            ),
            "XGBoost": XGBClassifier(
                n_estimators=300,
                max_depth=8,
                learning_rate=0.1,
                objective='multi:softmax',
                num_class=7,
                random_state=42
            )
        }

    def train_and_evaluate(self, X_train, X_test, y_train, y_test, label_encoder):

        results = {}

        for name, model in self.models.items():
            print(f"\nTraining {name}...")

            # Fit the model on the training data
            model.fit(X_train, y_train)

            # Make predictions on the test set
            y_pred = model.predict(X_test)

            # Calculate accuracy
            acc = accuracy_score(y_test, y_pred)

            print(f"\n{name} Accuracy: {acc:.4f}")
            # Print detailed classification report
            print(
                classification_report(
                    y_test,
                    y_pred,
                    target_names=label_encoder.classes_
                )
            )

            results[name] = acc

        return results